# Phase 13 — Evaluation Suite (Gemini-powered RAGAS)

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** run a slice of each dataset and confirm RAGAS uses **Gemini**, not OpenAI.

**👤 Optional:** `LANGSMITH_API_KEY` for tracing. No OpenAI key needed anywhere.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env')

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Confirm RAGAS is wired to Gemini (the critical gotcha)

In [ ]:
from app.llm.factory import get_llm, get_embeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
ragas_llm = LangchainLLMWrapper(get_llm(reasoning=True))   # gemini-3.5-flash
ragas_emb = LangchainEmbeddingsWrapper(get_embeddings())   # gemini-embedding-001
print('RAGAS LLM  ->', ragas_llm.langchain_llm.model)
print('RAGAS EMB  ->', ragas_emb.embeddings.model)
assert 'gemini' in ragas_llm.langchain_llm.model, 'RAGAS must use Gemini'

## 2. Run a small eval slice

In [ ]:
from evals.harness import run_dataset
results = run_dataset('evals/datasets/simple_queries.jsonl', limit=5)
from evals.metrics.answer_accuracy import score as acc_score
print('accuracy (5 items):', acc_score(results))

## 3. Routing + guardrail effectiveness

In [ ]:
from evals.metrics.routing_accuracy import score as route_score
from evals.metrics.guardrail_effectiveness import score as guard_score
print('routing :', route_score(results))
adv = run_dataset('evals/datasets/adversarial_queries.jsonl', limit=5)
print('guardrail block rate:', guard_score(adv))

## ✅ Acceptance check

In [ ]:
assert os.getenv('OPENAI_API_KEY') is None, 'No OpenAI key should be needed'
print('Phase 13 acceptance: PASS (RAGAS on Gemini; run `make eval` for the full report)')